Silver layer
From bronze to silver following are followed:

Created Dim tables -(Patient, Payers, Providers, Policies), Performed intial load, implemented merge logic
Created fact tables (Claims, CLiam_lines, payments)- performed load from bronze to silver

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze = "healthcare_adjudication.healthcare_adjudication_bronze"
silver = "healthcare_adjudication.healthcare_adjudication_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver}")


In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_patient AS target
USING (
    SELECT
        patient_id,
        name,
        gender,
        TO_DATE(dob,'yyyy-MM-dd') AS dob,
        state
    FROM healthcare_adjudication.healthcare_adjudication_bronze.patients
) AS source

ON target.patient_id = source.patient_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
    target.patient_name <> source.name
    OR target.gender <> source.gender
    OR target.state <> source.state
)

THEN UPDATE SET
    target.is_current = 'N',
    target.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    patient_id,
    patient_name,
    gender,
    dob,
    state,
    effective_start_date,
    effective_end_date,
    is_current
)

VALUES (
    source.patient_id,
    source.name,
    source.gender,
    source.dob,
    source.state,
    current_timestamp(),
    TIMESTAMP '9999-12-31',
    'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
  target.is_current = 'N',
  target.effective_end_date = current_date()

In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_provider AS target

USING (
SELECT
provider_id,
provider_name,
specialty,
state
FROM healthcare_adjudication.healthcare_adjudication_bronze.providers
) AS source

ON target.provider_id = source.provider_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
target.provider_name <> source.provider_name
OR target.specialty <> source.specialty
OR target.state <> source.state
)

THEN UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
provider_id,
provider_name,
specialty,
state,
effective_start_date,
effective_end_date,
is_current
)

VALUES (
source.provider_id,
source.provider_name,
source.specialty,
source.state,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
  target.is_current = 'N',
  target.effective_end_date = current_date()

In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_payer AS target

USING (
SELECT
payer_id,
payer_name,
plan_type
FROM healthcare_adjudication.healthcare_adjudication_bronze.payers
) AS source

ON target.payer_id = source.payer_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
target.payer_name <> source.payer_name
OR target.plan_type <> source.plan_type
)

THEN UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
payer_id,
payer_name,
plan_type,
effective_start_date,
effective_end_date,
is_current
)

VALUES (
source.payer_id,
source.payer_name,
source.plan_type,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_date()

In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_policy AS target

USING (

SELECT *
FROM (

SELECT
policy_id,
payer.payer_id AS payer_id,
coverage.plan AS plan,
coverage.limit AS coverage_limit,
coverage.copay AS copay,
ROW_NUMBER() OVER (PARTITION BY policy_id ORDER BY policy_id) AS rn

FROM healthcare_adjudication.healthcare_adjudication_bronze.policies

)

WHERE rn = 1

) source

ON target.policy_id = source.policy_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
target.payer_id <> source.payer_id
OR target.plan <> source.plan
OR target.coverage_limit <> source.coverage_limit
OR target.copay <> source.copay
)

THEN UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_timestamp()

WHEN NOT MATCHED

THEN INSERT (
policy_id,
payer_id,
plan,
coverage_limit,
copay,
effective_start_date,
effective_end_date,
is_current
)

VALUES (
source.policy_id,
source.payer_id,
source.plan,
source.coverage_limit,
source.copay,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
  target.is_current = 'N',
  target.effective_end_date = current_date()

In [0]:
%sql
INSERT INTO healthcare_adjudication.healthcare_adjudication_silver.silver_claims

SELECT
claim_id,
patient_id,
provider_id,
policy_id,
TO_DATE(claim_date,'yyyy-MM-dd') AS claim_date,
CAST(claim_amount AS DOUBLE),
status,
ingestion_time

FROM healthcare_adjudication.healthcare_adjudication_bronze.claims

In [0]:
%sql
INSERT INTO healthcare_adjudication.healthcare_adjudication_silver.silver_claim_lines

SELECT
claim_id,
procedure.code AS procedure_code,
procedure.charge AS line_amount,
ingestion_time

FROM healthcare_adjudication.healthcare_adjudication_bronze.claim_lines
LATERAL VIEW explode(procedures) p AS procedure

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_adjudication.healthcare_adjudication_silver.silver_payments (
    payment_id STRING,
    claim_id STRING,
    payer_id STRING,
    payment_amount DOUBLE,
    payment_status STRING,
    ingestion_time TIMESTAMP
)
USING DELTA